# 12_deep_learning_models

In [1]:
# 导入库
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# 设置路径并读取数据
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "module_F_sequence_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DIR / "train.csv"
test_path = PROCESSED_DIR / "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df.head()

train_df shape: (3583, 2)
test_df shape: (897, 2)


,sequence,label
0,SLLLNGGCKVSNYDE,1
1,DAEFRHDSGYEVHHQ,1
2,GRTGRGKPGIYRFVAPGE,1
3,ASLKPEFVQIINAKN,1
4,KCEFQDAYVLLSEKK,1


In [3]:
# 检查序列长度分布
train_df["seq_len"] = train_df["sequence"].apply(len)
test_df["seq_len"] = test_df["sequence"].apply(len)

print("Train length stats:")
print(train_df["seq_len"].describe())

print("\nTest length stats:")
print(test_df["seq_len"].describe())

Train length stats:
count    3583.000000
mean       16.334357
std         2.581094
min        11.000000
25%        15.000000
50%        15.000000
75%        18.000000
max        30.000000
Name: seq_len, dtype: float64

Test length stats:
count    897.000000
mean      16.434783
std        2.620837
min       11.000000
25%       15.000000
50%       15.000000
75%       19.000000
max       30.000000
Name: seq_len, dtype: float64


In [4]:
max_train_len = train_df["seq_len"].max()
max_test_len = test_df["seq_len"].max()
max_len = max(max_train_len, max_test_len)

print("max_train_len:", max_train_len)
print("max_test_len:", max_test_len)
print("global max_len:", max_len)

max_train_len: 30
max_test_len: 30
global max_len: 30


In [5]:
# 确定max_len
MAX_LEN = max_len
print("MAX_LEN =", MAX_LEN)

MAX_LEN = 30


In [6]:
# 定义氨基酸字母表和编码函数
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

print("Number of amino acids:", len(AMINO_ACIDS))
print(aa_to_idx)

Number of amino acids: 20
{'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19}


In [7]:
# 定义one-hot编码
def encode_sequence_onehot(seq, max_len=MAX_LEN):
    """
    将氨基酸序列编码为 shape = (max_len, 20) 的 one-hot 矩阵
    超过 max_len 的部分截断，不足部分补 0
    """
    encoded = np.zeros((max_len, len(AMINO_ACIDS)), dtype=np.float32)

    for i, aa in enumerate(seq[:max_len]):
        if aa in aa_to_idx:
            encoded[i, aa_to_idx[aa]] = 1.0

    return encoded

In [8]:
# 利用one-hot编码整个数据集
X_train_seq = np.array([encode_sequence_onehot(seq, MAX_LEN) for seq in train_df["sequence"]])
X_test_seq = np.array([encode_sequence_onehot(seq, MAX_LEN) for seq in test_df["sequence"]])

y_train_full = train_df["label"].values.astype(np.float32)
y_test = test_df["label"].values.astype(np.float32)

print("X_train_seq:", X_train_seq.shape)
print("X_test_seq:", X_test_seq.shape)
print("y_train_full:", y_train_full.shape)
print("y_test:", y_test.shape)

X_train_seq: (3583, 30, 20)
X_test_seq: (897, 30, 20)
y_train_full: (3583,)
y_test: (897,)


In [9]:
# 划分 train / val
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_seq,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

print("X_tr:", X_tr.shape)
print("X_val:", X_val.shape)
print("X_test_seq:", X_test_seq.shape)

X_tr: (2866, 30, 20)
X_val: (717, 30, 20)
X_test_seq: (897, 30, 20)


In [10]:
# 写 PyTorch Dataset
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [11]:
# 构建DataLoader
BATCH_SIZE = 128

train_dataset = SequenceDataset(X_tr, y_tr)
val_dataset = SequenceDataset(X_val, y_val)
test_dataset = SequenceDataset(X_test_seq, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

train batches: 23
val batches: 6
test batches: 8


In [12]:
# 定义评估函数
def compute_binary_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "ACC": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall_SN": sensitivity,
        "Specificity_SP": specificity,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }
    return metrics

## （1）CNN

### 1 定义CNN模型

In [31]:
class CNNEmbedding(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=3, padding=1)

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.fc = nn.Linear(256, 1)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        # x: (batch, seq_len)  ← 注意这里变了

        x = self.embedding(x)           # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)          # (batch, embed_dim, seq_len)

        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))

        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze(1)

In [32]:
# 设置device、loss、优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)

print("pos_weight:", pos_weight.item())

Using device: cpu
pos_weight: 1.6245421171188354


In [33]:
cnn_model = CNN1DClassifier(input_dim=20).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-4)

### 2 写训练和验证函数

In [24]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(dataloader.dataset)

In [25]:
@torch.no_grad()
def predict_proba(model, dataloader, device):
    model.eval()

    all_probs = []
    all_labels = []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs.tolist())
        all_labels.extend(y_batch.numpy().tolist())

    return np.array(all_labels), np.array(all_probs)

In [26]:
def evaluate_dl_model(model, dataloader, device, threshold=0.5):
    y_true, y_prob = predict_proba(model, dataloader, device)
    y_pred = (y_prob >= threshold).astype(int)
    return compute_binary_metrics(y_true, y_pred, y_prob)

### 3 先训练一个 20 epoch 的 CNN baseline

In [35]:
EPOCHS = 250

train_losses = []
val_mccs = []
best_val_mcc = -np.inf
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(cnn_model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_dl_model(cnn_model, val_loader, device, threshold=0.5)

    train_losses.append(train_loss)
    val_mccs.append(val_metrics["MCC"])

    if val_metrics["MCC"] > best_val_mcc:
        best_val_mcc = val_metrics["MCC"]
        best_state_dict = {k: v.cpu().clone() for k, v in cnn_model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val MCC: {val_metrics['MCC']:.4f} | "
        f"Val ACC: {val_metrics['ACC']:.4f} | "
        f"Val AUC: {val_metrics['ROC_AUC']:.4f}"
    )

Epoch 01 | Train Loss: 0.3788 | Val MCC: 0.2702 | Val ACC: 0.6513 | Val AUC: 0.6795
Epoch 02 | Train Loss: 0.3960 | Val MCC: 0.2624 | Val ACC: 0.6541 | Val AUC: 0.6795
Epoch 03 | Train Loss: 0.3861 | Val MCC: 0.2561 | Val ACC: 0.6499 | Val AUC: 0.6814
Epoch 04 | Train Loss: 0.3817 | Val MCC: 0.2650 | Val ACC: 0.6583 | Val AUC: 0.6808
Epoch 05 | Train Loss: 0.3738 | Val MCC: 0.2690 | Val ACC: 0.6555 | Val AUC: 0.6809
Epoch 06 | Train Loss: 0.3775 | Val MCC: 0.2720 | Val ACC: 0.6611 | Val AUC: 0.6807
Epoch 07 | Train Loss: 0.3734 | Val MCC: 0.2637 | Val ACC: 0.6611 | Val AUC: 0.6798
Epoch 08 | Train Loss: 0.3660 | Val MCC: 0.2690 | Val ACC: 0.6555 | Val AUC: 0.6799
Epoch 09 | Train Loss: 0.3593 | Val MCC: 0.2608 | Val ACC: 0.6402 | Val AUC: 0.6814
Epoch 10 | Train Loss: 0.3618 | Val MCC: 0.2715 | Val ACC: 0.6527 | Val AUC: 0.6801
Epoch 11 | Train Loss: 0.3582 | Val MCC: 0.2659 | Val ACC: 0.6555 | Val AUC: 0.6797
Epoch 12 | Train Loss: 0.3418 | Val MCC: 0.2770 | Val ACC: 0.6527 | Val AUC:

In [36]:
cnn_model.load_state_dict(best_state_dict)

cnn_val_metrics = evaluate_dl_model(cnn_model, val_loader, device, threshold=0.5)
cnn_test_metrics = evaluate_dl_model(cnn_model, test_loader, device, threshold=0.5)

print("CNN validation metrics:")
display(pd.DataFrame([cnn_val_metrics], index=["CNN_Val"]))

print("CNN test metrics:")
display(pd.DataFrame([cnn_test_metrics], index=["CNN_Test"]))

CNN validation metrics:


,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
CNN_Val,0.670851,0.568773,0.56044,0.738739,0.564576,0.300046,0.6906,0.531242,153,328,116,120


CNN test metrics:


,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
CNN_Test,0.662207,0.556196,0.564327,0.722523,0.560232,0.286067,0.688257,0.532822,193,401,154,149


## （2）BiLSTM

In [37]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 留给 padding
idx_to_aa = {i + 1: aa for i, aa in enumerate(AMINO_ACIDS)}

PAD_IDX = 0
VOCAB_SIZE = len(AMINO_ACIDS) + 1  # +1 because 0 is padding

print("VOCAB_SIZE:", VOCAB_SIZE)
print("PAD_IDX:", PAD_IDX)
print(aa_to_idx)

VOCAB_SIZE: 21
PAD_IDX: 0
{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20}


In [38]:
def encode_sequence_index(seq, max_len=MAX_LEN):
    """
    将氨基酸序列编码为长度为 max_len 的整数序列
    padding位置为0
    """
    encoded = np.zeros(max_len, dtype=np.int64)

    for i, aa in enumerate(seq[:max_len]):
        encoded[i] = aa_to_idx.get(aa, 0)

    return encoded

In [39]:
example_seq = train_df.loc[0, "sequence"]
example_index = encode_sequence_index(example_seq, max_len=MAX_LEN)

print("Example sequence:", example_seq)
print("Encoded index shape:", example_index.shape)
print(example_index[:20])

Example sequence: SLLLNGGCKVSNYDE
Encoded index shape: (30,)
[16 10 10 10 12  6  6  2  9 18 16 12 20  3  4  0  0  0  0  0]


In [40]:
X_train_idx = np.array([encode_sequence_index(seq, MAX_LEN) for seq in train_df["sequence"]])
X_test_idx = np.array([encode_sequence_index(seq, MAX_LEN) for seq in test_df["sequence"]])

y_train_full = train_df["label"].values.astype(np.float32)
y_test = test_df["label"].values.astype(np.float32)

print("X_train_idx:", X_train_idx.shape)
print("X_test_idx:", X_test_idx.shape)
print("y_train_full:", y_train_full.shape)
print("y_test:", y_test.shape)

X_train_idx: (3583, 30)
X_test_idx: (897, 30)
y_train_full: (3583,)
y_test: (897,)


In [41]:
X_tr_idx, X_val_idx, y_tr, y_val = train_test_split(
    X_train_idx,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

print("X_tr_idx:", X_tr_idx.shape)
print("X_val_idx:", X_val_idx.shape)
print("X_test_idx:", X_test_idx.shape)

X_tr_idx: (2866, 30)
X_val_idx: (717, 30)
X_test_idx: (897, 30)


In [42]:
class SequenceIndexDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)   # 注意这里是 long
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [49]:
BATCH_SIZE = 128

train_dataset_lstm = SequenceIndexDataset(X_tr_idx, y_tr)
val_dataset_lstm = SequenceIndexDataset(X_val_idx, y_val)
test_dataset_lstm = SequenceIndexDataset(X_test_idx, y_test)

train_loader_lstm = DataLoader(train_dataset_lstm, batch_size=BATCH_SIZE, shuffle=True)
val_loader_lstm = DataLoader(val_dataset_lstm, batch_size=BATCH_SIZE, shuffle=False)
test_loader_lstm = DataLoader(test_dataset_lstm, batch_size=BATCH_SIZE, shuffle=False)

print("train batches:", len(train_loader_lstm))
print("val batches:", len(val_loader_lstm))
print("test batches:", len(test_loader_lstm))

train batches: 23
val batches: 6
test batches: 8


In [50]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        embed_dim=64,
        hidden_dim=128,
        num_layers=1,
        dropout=0.3,
        pad_idx=PAD_IDX
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0 if num_layers == 1 else dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x)  # -> (batch, seq_len, embed_dim)

        lstm_out, (h_n, c_n) = self.lstm(x)

        # 双向LSTM最后一层的正向和反向hidden state
        forward_last = h_n[-2]   # (batch, hidden_dim)
        backward_last = h_n[-1]  # (batch, hidden_dim)

        h = torch.cat([forward_last, backward_last], dim=1)  # (batch, hidden_dim*2)

        h = self.dropout(h)
        h = self.relu(self.fc1(h))
        logits = self.fc2(h).squeeze(1)

        return logits

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)

print("pos_weight:", pos_weight.item())

Using device: cpu
pos_weight: 1.6245421171188354


In [51]:
bilstm_model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=64,
    hidden_dim=256,
    num_layers=1,
    dropout=0.3,
    pad_idx=PAD_IDX
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-4)

In [53]:
EPOCHS = 20

train_losses_lstm = []
val_mccs_lstm = []
best_val_mcc_lstm = -np.inf
best_state_dict_lstm = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        bilstm_model,
        train_loader_lstm,
        criterion,
        optimizer,
        device
    )

    val_metrics = evaluate_dl_model(
        bilstm_model,
        val_loader_lstm,
        device,
        threshold=0.5
    )

    train_losses_lstm.append(train_loss)
    val_mccs_lstm.append(val_metrics["MCC"])

    if val_metrics["MCC"] > best_val_mcc_lstm:
        best_val_mcc_lstm = val_metrics["MCC"]
        best_state_dict_lstm = {k: v.cpu().clone() for k, v in bilstm_model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val MCC: {val_metrics['MCC']:.4f} | "
        f"Val ACC: {val_metrics['ACC']:.4f} | "
        f"Val AUC: {val_metrics['ROC_AUC']:.4f}"
    )

Epoch 01 | Train Loss: 0.0475 | Val MCC: 0.1494 | Val ACC: 0.6011 | Val AUC: 0.5971
Epoch 02 | Train Loss: 0.0328 | Val MCC: 0.1577 | Val ACC: 0.6011 | Val AUC: 0.5946
Epoch 03 | Train Loss: 0.0206 | Val MCC: 0.1685 | Val ACC: 0.6025 | Val AUC: 0.6050
Epoch 04 | Train Loss: 0.0132 | Val MCC: 0.1497 | Val ACC: 0.5941 | Val AUC: 0.5980
Epoch 05 | Train Loss: 0.0108 | Val MCC: 0.1601 | Val ACC: 0.6025 | Val AUC: 0.5964
Epoch 06 | Train Loss: 0.0090 | Val MCC: 0.1530 | Val ACC: 0.5997 | Val AUC: 0.5965
Epoch 07 | Train Loss: 0.0077 | Val MCC: 0.1648 | Val ACC: 0.6039 | Val AUC: 0.5969
Epoch 08 | Train Loss: 0.0069 | Val MCC: 0.1660 | Val ACC: 0.6053 | Val AUC: 0.5976
Epoch 09 | Train Loss: 0.0066 | Val MCC: 0.1778 | Val ACC: 0.6081 | Val AUC: 0.5981
Epoch 10 | Train Loss: 0.0059 | Val MCC: 0.1600 | Val ACC: 0.6039 | Val AUC: 0.5936
Epoch 11 | Train Loss: 0.0066 | Val MCC: 0.1661 | Val ACC: 0.6025 | Val AUC: 0.5939
Epoch 12 | Train Loss: 0.0056 | Val MCC: 0.1648 | Val ACC: 0.6053 | Val AUC:

In [54]:
bilstm_model.load_state_dict(best_state_dict_lstm)

bilstm_val_metrics = evaluate_dl_model(bilstm_model, val_loader_lstm, device, threshold=0.5)
bilstm_test_metrics = evaluate_dl_model(bilstm_model, test_loader_lstm, device, threshold=0.5)

print("BiLSTM validation metrics:")
display(pd.DataFrame([bilstm_val_metrics], index=["BiLSTM_Val"]))

print("BiLSTM test metrics:")
display(pd.DataFrame([bilstm_test_metrics], index=["BiLSTM_Test"]))

BiLSTM validation metrics:


,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
BiLSTM_Val,0.608089,0.486111,0.512821,0.666667,0.499109,0.177779,0.598051,0.455472,140,296,148,133


BiLSTM test metrics:


,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
BiLSTM_Test,0.620959,0.502747,0.535088,0.673874,0.518414,0.206686,0.612942,0.485133,183,374,181,159
